<a href="https://colab.research.google.com/github/SarveshMD/colab-notebooks/blob/main/nlp_imdb_distilbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import copy
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.metrics import accuracy_score
import numpy as np

In [2]:
import os
# Force the script to only see the first T4 GPU, making it a simple single-GPU setup!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # Should say 'cuda' cleanly

Using device: cuda


In [3]:
raw_datasets = load_dataset("stanfordnlp/imdb")
print(raw_datasets.shape, type(raw_datasets))

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

{'train': (25000, 2), 'test': (25000, 2), 'unsupervised': (50000, 2)} <class 'datasets.dataset_dict.DatasetDict'>


In [4]:
raw_datasets["train"][raw_datasets["train"].shape[0]-1]

{'text': 'The story centers around Barry McKenzie who must go to England if he wishes to claim his inheritance. Being about the grossest Aussie shearer ever to set foot outside this great Nation of ours there is something of a culture clash and much fun and games ensue. The songs of Barry McKenzie(Barry Crocker) are highlights.',
 'label': 1}

In [5]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_datasets = raw_datasets.map(tokenize_fn, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [6]:
tokenized_datasets.keys()

tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

In [7]:
train_dataset = tokenized_datasets["train"].shuffle(seed=13)
test_dataset = tokenized_datasets["test"].shuffle(seed=13)

In [8]:
print(train_dataset)
print(test_dataset)

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})


In [9]:
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=64, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=64, num_workers=4)

In [10]:
next(iter(train_loader))

{'labels': tensor([1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0,
         1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0,
         0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0]),
 'input_ids': tensor([[  101,  1045,  3427,  ...,     0,     0,     0],
         [  101,  1996,  2034,  ...,     0,     0,     0],
         [  101, 11951,  1999,  ...,     0,     0,     0],
         ...,
         [  101,  1045,  2031,  ...,     0,     0,     0],
         [  101,  1045,  3427,  ...,     0,     0,     0],
         [  101,  1045,  2001,  ...,     0,     0,     0]]),
 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
     

In [11]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

model.to(device)

print(device, next(model.parameters()).device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


cuda cuda:0


In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=5e-5)
epochs = 3

for epoch in range(epochs):
    model.train()
    total_training_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()

        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_training_loss += loss.item()

    model.eval()
    all_preds = []
    all_labels = []
    total_test_loss = 0.0

    with torch.inference_mode():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss

            total_test_loss += loss.item()

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)

            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())

    avg_training_loss = total_training_loss / len(train_loader)
    avg_test_loss = total_test_loss / len(test_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"\nEpoch {epoch+1}/{epochs} finished. | Avg. Training Loss: {avg_training_loss:.4f} | Avg. Testing Loss: {avg_test_loss:.4f} | Testing Accuracy: {accuracy*100:.2f}%")


Epoch 1/3 finished. | Avg. Training Loss: 0.2731 | Avg. Testing Loss: 0.1922 | Testing Accuracy: 92.56

Epoch 2/3 finished. | Avg. Training Loss: 0.1596 | Avg. Testing Loss: 0.1789 | Testing Accuracy: 93.11


KeyboardInterrupt: 

## Custom Reviews (from letterboxd.com and reddit.com)

* I have to cut some slack for the model. The 4th - 6th examples are primarily negative and flip the switch near the end.
* matt damon: *eats a boiled potato* me: now that's cinema
* me pretending to be a tornado so glen powell will chase me
* for the above two, again, the training data is mostly "movie review" kind. not the letterboxd meme culture kind. so I don't blame the model for being wrong.
* I'm surprised how good it is at predicting negative reviews from the custom tests.

In [13]:
custom_reviews = [
    "i still remember the specific feeling of being 10 years old and watching this for the first time. i thought 'this is the best movie i’ve ever seen in my life' and that’s still true",
    "Iconic",
    "Oh. My. God. I have a headache, but it's the best headache I've ever had.",
    "This is the only movie i’ve watched more than once, and I’ve probably seen it close to 10 times. I have a difficult time understanding sci-fi movies, as most of the time the logic behind what they’re doing is very flawed and immediately takes me out of whatever world they’re trying to build, but not Interstellar.",
    "This is the only movie i’ve watched more than once, and I’ve probably seen it close to 10 times.",
    "I have a difficult time understanding sci-fi movies, as most of the time the logic behind what they’re doing is very flawed and immediately takes me out of whatever world they’re trying to build, but not this one.",
    "matt damon: *eats a boiled potato* me: now that's cinema",
    "all I can say is... amaze, amaze, amaze.",
    "Yeah the flying is amazing and it has the most thrilling action finale I’ve seen in a blockbuster in years but honestly one of the most exciting things about this movie is that Tom Cruise finally made a movie that acknowledges he’s getting older",
    "me pretending to be a tornado so glen powell will chase me",
    "Sorry but it’s kind of almost a completely perfect concert film (so clean, so classy, zips along knowing exactly what it needs to do, nothing more, nothing less) and I said “she’s so good” out loud more times than I could actually control, generational imo",
    "Olivia I la-la-la-la-la-la love you",
    "now that’s what I call a comfort movie",
    # negative
    "A heavy, inescapable, and oppressive inferiority complex suffocates the entire runtime of this film, making it miserable and plain unfun to watch.",
    "this might be the first film i’ve ever seen that actually offended me. and i’ve seen a lot of shit. i’ve never watched anything so blatantly regressive. there’s absolutely no narrative payoff for any of the antagonists. in fact, most of them end up better off than when they were introduced. the humour, wherever you can find it, revolves around the whiteness. how great it is to be white, and why you should aspire to become white. it’s enjoyable and clever at the start and then after a while it becomes uncomfortable and repetitive.",
    "this was so bad wth. the book wasn’t THAT great to begin with but they really gave us a new low with this.",
    "I don’t know if it was the script, direction or what but both Bacon and Seyfried are just brutal in this. Zero chemistry. The little girl isn’t helping matters either. Thought it had some cool ideas near the end but it took way too long to get there and by that time I was mostly checked out.",
    "it rlly hurts to see good actors in horrible movies",
    "emma stone is the only redeeming quality about this movie like i literally don’t even want to talk about it. it’s a miracle i got through this entire thing. i felt guilty for watching so many banger movies in succession that i felt like i needed to even it out with netflix garbage and other awful pictures. life is too short to watch shit, going back to bangers right after i give the gray man a chance next. you can’t have stone without gosling!",
    "This movie doesn't scrape the bottom of the barrel. This movie isn't the bottom of the barrel. This movie isn't below the bottom of the barrel. This movie doesn't deserve to be mentioned in the same sentence with barrels.",
    "I hated this movie. Hated hated hated hated hated this movie. Hated it. Hated every simpering stupid vacant audience-insulting moment of it. Hated the sensibility that thought anyone would like it. Hated the implied insult to the audience by its belief that anyone would be entertained by it."
]

custom_labels = [1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0]

assert len(custom_reviews) == len(custom_labels)

In [15]:
model.eval()
inputs = tokenizer(custom_reviews, padding=True, truncation=True, return_tensors="pt")

with torch.inference_mode():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)

logits = outputs.logits
predictions = torch.argmax(logits, dim=-1)
print(predictions)
for indx, pred in enumerate(predictions):
    print(f"Review: {custom_reviews[indx]}\nTrue Label: {'Positive' if custom_labels[indx] == 1 else 'Negative'} | Predicted Label: {'Positive' if predictions[indx].item() == 1 else 'Negative'}\n")

accuracy_custom_data = accuracy_score(custom_labels, predictions.cpu().numpy()) * 100
print(f"Accuracy on Custom Data: {accuracy_custom_data:.2f}%")

tensor([1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       device='cuda:0')
Review: i still remember the specific feeling of being 10 years old and watching this for the first time. i thought 'this is the best movie i’ve ever seen in my life' and that’s still true
True Label: Positive | Predicted Label: Positive

Review: Iconic
True Label: Positive | Predicted Label: Positive

Review: Oh. My. God. I have a headache, but it's the best headache I've ever had.
True Label: Positive | Predicted Label: Positive

Review: This is the only movie i’ve watched more than once, and I’ve probably seen it close to 10 times. I have a difficult time understanding sci-fi movies, as most of the time the logic behind what they’re doing is very flawed and immediately takes me out of whatever world they’re trying to build, but not Interstellar.
True Label: Positive | Predicted Label: Negative

Review: This is the only movie i’ve watched more than once, and I’ve probably seen it close to 

### Challenging ChatGPT generated negative reviews
* Heavy sarcasm with a lot of positive key words that fool the model into thinking it's a positive review

In [16]:
more_negative_chatgpt_generated = [
    "What a masterpiece. It takes real talent to make two hours feel like an entire weekend.",
    "The acting was incredible; I've never seen emotions avoided so consistently.",
    "A truly unforgettable experience, despite my best efforts to forget it.",
    "The plot was wonderfully unpredictable because it seemed unaware of itself.",
    "An excellent cure for insomnia. I haven't slept that peacefully in years.",
    "The dialogue was refreshingly unique; no actual person would ever say those lines.",
    "The pacing was fantastic if the goal was to make time move backwards.",
    "A bold and innovative approach to storytelling: replacing logic with confidence.",
    "The ending exceeded my expectations by somehow making even less sense than the rest.",
    "I admire the filmmakers' commitment to proving that a big budget can't buy a good script."
]
more_labels = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

model.eval()
inputs = tokenizer(more_negative_chatgpt_generated, padding=True, truncation=True, return_tensors="pt")

with torch.inference_mode():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)

logits = outputs.logits
predictions = torch.argmax(logits, dim=-1)
print(predictions)
for indx, pred in enumerate(predictions):
    print(f"Review: {more_negative_chatgpt_generated[indx]}\nTrue Label: {'Positive' if more_labels[indx] == 1 else 'Negative'} | Predicted Label: {'Positive' if predictions[indx].item() == 1 else 'Negative'}\n")

accuracy_custom_data = accuracy_score(more_labels, predictions.cpu().numpy()) * 100
print(f"Accuracy on Custom Data: {accuracy_custom_data:.2f}%")

tensor([1, 1, 1, 1, 1, 1, 1, 1, 0, 1], device='cuda:0')
Review: What a masterpiece. It takes real talent to make two hours feel like an entire weekend.
True Label: Negative | Predicted Label: Positive

Review: The acting was incredible; I've never seen emotions avoided so consistently.
True Label: Negative | Predicted Label: Positive

Review: A truly unforgettable experience, despite my best efforts to forget it.
True Label: Negative | Predicted Label: Positive

Review: The plot was wonderfully unpredictable because it seemed unaware of itself.
True Label: Negative | Predicted Label: Positive

Review: An excellent cure for insomnia. I haven't slept that peacefully in years.
True Label: Negative | Predicted Label: Positive

Review: The dialogue was refreshingly unique; no actual person would ever say those lines.
True Label: Negative | Predicted Label: Positive

Review: The pacing was fantastic if the goal was to make time move backwards.
True Label: Negative | Predicted Label: Positive

In [17]:
torch.save(model.state_dict(), "imdb_distilbert.pth")